# Base Agent
`baseAgent.py` provides the shared construction interface for all MAF specialist agents.
To see this implemented, see `adviceAgent.ipynb`.

## AG2 → MAF Migration Summary
| AG2 Pattern | MAF Equivalent |
|---|---|
| `autogen.LLMConfig` + `OAI_CONFIG_LIST.json` | `OpenAIChatClient` with explicit `base_url`, `model`, `api_key` |
| `langchain.tools.BaseTool` | `@tool`-decorated callable (MAF-native) |
| `generate_llm_config(tool)` | Removed — MAF introspects tool schema from type annotations |
| `getToolsConfig()` | Removed — no manual JSON schema serialization needed |
| `getConfig()` / `LLMConfig(tools=..., config_list=...)` | Removed — client constructed once in `BaseAgent.__init__` |
| `ConversableAgent(human_input_mode="NEVER")` | `Agent(client=..., name=..., instructions=..., tools=...)` |
| `registerExecution(user_proxy)` | Removed — MAF agents execute tools internally, no proxy needed |
| `user_proxy.register_for_execution(name=...)(fn)` | Removed — tools registered at `Agent` construction via `tools=[...]` |

## Imports
NOTE: We are using Microsoft Agent Framework (MAF) `1.x`. All `autogen` and `langchain` imports are replaced.
The `OpenAIChatClient` is used here since the backend is an Ollama proxy with an OpenAI-compatible API.
If migrating to Azure AI Foundry, swap `OpenAIChatClient` for `FoundryChatClient` with `AzureCliCredential`.

In [ ]:
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient
from typing import List, Any

## Shared Client
**AG2 equivalent:** `LLMConfig.from_json(path="OAI_CONFIG_LIST.json")`

In AG2, `OAI_CONFIG_LIST.json` was loaded from disk and passed into every agent via `getConfig()`.
This meant config was coupled to a file on disk and had to be re-read or re-assigned for each agent.

In MAF, the client is a single shared instance constructed once.
All specialist agents that subclass or use `BaseAgent` share this client automatically.
Changing the model or endpoint here propagates everywhere with no additional work.

Config values from `OAI_CONFIG_LIST.json` map directly to constructor arguments:
- `model` → `model=`
- `base_url` → `base_url=`
- `api_key` → `api_key=`
- `temperature`, `tool_choice` → `default_options={...}`

**`OAI_CONFIG_LIST.json` is no longer needed and can be deleted.**

In [ ]:
_client = OpenAIChatClient(
    model="qwen2.5:14b",
    base_url="https://proxy/v1",
    api_key="ollama",
    default_options={"temperature": 0.3, "tool_choice": "required"},
)

## Context Injection Helper
**AG2 equivalent:** `UpdateSystemMessage` with `{user_name}`, `{query_history}`, `{response_history}`, `{system_instructions}` template vars

In AG2, `UpdateSystemMessage` was a stateful hook that rewrote the agent's system prompt before each reply.
This worked but was implicit — the rewrite happened inside the framework loop, not in visible user code.

In MAF, context injection is explicit. `build_instructions()` takes the base instructions string and a runtime
context dict, and returns a merged instructions string. This is called in `BaseAgent.with_context()` to produce
a per-request agent instance with enriched instructions.

This approach is:
- **Stateless** — no shared mutable state between requests
- **Testable** — `build_instructions()` is a pure function
- **Explicit** — the merged prompt is visible Python, not a framework side effect

In [ ]:
def build_instructions(base_instructions: str, context: dict | None = None) -> str:
    """
    Merges base system instructions with runtime context variables.

    context keys (all optional):
        user_name           : str
        query_history       : list[str]
        response_history    : list[str]
        system_instructions : list[str]
    """
    if not context:
        return base_instructions

    user_name           = context.get("user_name", "the user")
    query_history       = "\n".join(context.get("query_history", []))
    response_history    = "\n".join(context.get("response_history", []))
    system_instructions = "\n".join(context.get("system_instructions", []))

    context_block = (
        f"\n\n---\n"
        f"You are helping {user_name}. "
        f"This is the only user you are allowed to provide information for. "
        f"NEVER give this user the information of any other user.\n\n"
        f"QUERY HISTORY (user messages for context):\n{query_history}\n\n"
        f"RESPONSE HISTORY (assistant messages for context):\n{response_history}\n\n"
        f"SYSTEM INSTRUCTIONS (critical — always follow):\n{system_instructions}\n"
        f"---"
    )

    return base_instructions + context_block

## BaseAgent Class
The main focus when extending this class is to have a proper `name`, `description`, `instructions`, and `tools`.
Everything else is handled automatically.

### What Changed from AG2

**`getToolsConfig()` → removed**
In AG2, `getToolsConfig()` manually serialized each LangChain `BaseTool`'s Pydantic schema into a JSON blob
that AutoGen could pass to the LLM as a function definition. This was necessary because AG2 didn't know how
to read LangChain tools natively.

In MAF, tools are `@tool`-decorated callables. MAF introspects their type annotations and docstrings directly
to generate the function schema — no manual JSON serialization needed.

**`getConfig()` → removed**
In AG2, `getConfig()` read `OAI_CONFIG_LIST.json` from disk, merged in the tool schemas, and built an
`LLMConfig` object. Every agent had to do this separately.

In MAF, the shared `_client` instance above handles all of this. No per-agent config construction needed.

**`createAgent()` → `_create_agent()`**
In AG2, `createAgent()` instantiated a `ConversableAgent` with `human_input_mode="NEVER"`.
We set `human_input_mode="NEVER"` because the orchestrator handles all user interaction — if a specialist
agent prompted for input, it would crash the FastAPI web UI.

In MAF, `Agent` has no `human_input_mode` — agents never prompt for input by design.
The equivalent safety guarantee is structural, not a flag you have to remember to set.

**`registerExecution(user_proxy)` → removed**
This was the most ceremonial part of AG2 agent setup. For tool execution to work, two things had to happen:
1. The tool had to be registered on the agent via `generate_llm_config`
2. A `UserProxyAgent` had to explicitly grant execution permission via `register_for_execution()`

This two-step dance existed because in AG2, the LLM agent *proposed* tool calls but a separate proxy *executed* them.
In MAF, the agent handles the full loop internally. Tools passed to `tools=[...]` at construction are
called directly by the agent during its run — no proxy, no permission grant, no `registerExecution` call.

**`with_context()` → new**
Returns a per-request agent with runtime context injected into its instructions.
Replaces AG2's stateful `UpdateSystemMessage` hook with a stateless, explicit alternative.
See the Context Injection Helper cell above for details.

In [ ]:
class BaseAgent:
    def __init__(
        self,
        name: str,
        description: str,
        instructions: str,
        tools: List[Any],
    ):
        self.name         = name
        self.description  = description
        self.instructions = instructions
        self.tools        = tools
        self.agent        = self._create_agent()

    def _create_agent(self) -> Agent:
        return _client.as_agent(
            name=self.name,
            instructions=self.instructions,
            tools=self.tools,
        )

    def with_context(self, context: dict) -> Agent:
        """
        Returns a context-enriched agent instance for a single request.
        Rebuilds instructions with runtime context injected.

        AG2 equivalent: UpdateSystemMessage hook (stateful, implicit)
        MAF equivalent: build_instructions() + new Agent instance (stateless, explicit)

        Usage in orchestration.py:
            enriched = db_agent_wrapper.with_context(ctx)
            result = await enriched.run(task)
        """
        enriched_instructions = build_instructions(self.instructions, context)
        return _client.as_agent(
            name=self.name,
            instructions=enriched_instructions,
            tools=self.tools,
        )

## Tool Requirements
In AG2, tools were LangChain `BaseTool` subclasses. The `generate_llm_config()` function read their
`args_schema` Pydantic model and serialized it manually into a JSON function definition.

In MAF, tools must be `@tool`-decorated callables. MAF reads type annotations and the docstring automatically.

**Minimum tool structure for MAF:**
```python
from agent_framework import tool

@tool
def my_tool(param_one: str, param_two: int) -> str:
    """One-line description — this becomes the tool's description for the LLM."""
    # implementation
    return result
```

For tools that need dependency injection (e.g. `DatabaseConnection`, `DockerCodeExecutor`),
instantiate the tool class first, then pass the bound instance to `tools=[...]`:
```python
tools = [
    QueryPatientInfoTool(db_connection),   # __call__ is @tool-decorated
    UpdatePatientRecordTool(db_connection, models),
]
```
The `@tool` decorator wraps `__call__`, not `__init__`, so constructor injection survives intact.

See `docs/tools/baseApiTool.ipynb` for tool implementation details.